
## Folder 05 — File 05: Quarantine Management


The workflow:

- Read Silver datasets.
- Identify invalid records.
- Attach quarantine reasons.
- Separate valid and invalid records.
- Store invalid records in a centralized Delta table.
- Generate a quarantine summary.
- Inspect quarantined records.


### Quarantine Architecture:


    ```text
    ┌──────────────────────────────────────────────┐
    │              SILVER DATA TABLES              │
    │                                              │
    │     Customers · Products · Orders            │
    │     Order Items · Payments · Shipments       │
    │     Inventory Snapshots                      │
    └──────────────────────────────────────────────┘
                        │
                        ▼
    ┌──────────────────────────────────────────────┐
    │           DATA QUALITY VALIDATION            │
    │                                              │
    │        Valid Records / Invalid Records       │
    └──────────────────────────────────────────────┘
                        │
                        ▼
    ┌──────────────────────────────────────────────┐
    │              QUARANTINE TABLES               │
    │                                              │
    │  ┌────────────────────────────────────────┐  │
    │  │ quarantine_records                    │  │
    │  │ Latest execution's invalid records    │  │
    │  └────────────────────────────────────────┘  │
    │                                              │
    │  ┌────────────────────────────────────────┐  │
    │  │ quarantine_history                     │  │
    │  │ Historical invalid records             │  │
    │  └────────────────────────────────────────┘  │
    │                                              │
    │  ┌────────────────────────────────────────┐  │
    │  │ quarantine_summary                     │  │
    │  │ Failure counts by dataset and reason   │  │
    │  └────────────────────────────────────────┘  │
    └──────────────────────────────────────────────┘
    ```


In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

from datetime import datetime
import uuid

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"

QUARANTINE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_records"
QUARANTINE_HISTORY_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_history"
VALIDATION_SUMMARY_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_summary"
DATASET_STATISTICS_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.quarantine_dataset_statistics"

DATASETS = {
    "customers": {
        "table": "silver_customers",
        "business_key": "customer_id"
    },
    "products": {
        "table": "silver_products",
        "business_key": "product_id"
    },
    "orders": {
        "table": "silver_orders",
        "business_key": "order_id"
    },
    "order_items": {
        "table": "silver_order_items",
        "business_key": "order_item_id"
    },
    "payments": {
        "table": "silver_payments",
        "business_key": "payment_id"
    },
    "shipments": {
        "table": "silver_shipments",
        "business_key": "shipment_id"
    },
    "inventory": {
        "table": "silver_inventory",
        "business_key": "inventory_snapshot_id"
    }
}

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow()

print(f"Run ID: {RUN_ID}")
print(f"Run timestamp (UTC): {RUN_TIMESTAMP}")


## 3. Create schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}"
)

print(f"Schema available: {CATALOG}.{SILVER_SCHEMA}")


## 4. Define reusable validation function

def identify_invalid_records(
    df,
    dataset_name,
    validation_rules
):
    """
    Adds all matching rule names to _quarantine_reason.
    Returns (valid_df, invalid_df).
    """

    required_metadata_columns = [
        "_quarantine_reason",
        "_quarantine_dataset",
        "_quarantined_at",
        "_quarantine_run_timestamp",
        "_quarantine_run_id"
    ]

    # Avoid accidental duplicate metadata columns if the notebook is

    # run against a source that already contains these fields.

    for column_name in required_metadata_columns:
        if column_name in df.columns:
            df = df.drop(column_name)

    invalid_conditions = []

    for rule_name, condition in validation_rules:
        invalid_conditions.append(
            F.when(condition, F.lit(rule_name))
        )

    result_df = df.withColumn(
        "_quarantine_reason",
        F.concat_ws(" | ", *invalid_conditions)
    )

    invalid_df = (
        result_df
        .filter(F.length(F.col("_quarantine_reason")) > 0)
        .withColumn("_quarantine_dataset", F.lit(dataset_name))
        .withColumn("_quarantined_at", F.current_timestamp())
        .withColumn(
            "_quarantine_run_timestamp",
            F.lit(RUN_TIMESTAMP).cast("timestamp")
        )
        .withColumn("_quarantine_run_id", F.lit(RUN_ID))
    )

    valid_df = (
        result_df
        .filter(F.length(F.col("_quarantine_reason")) == 0)
        .drop("_quarantine_reason")
    )

    return valid_df, invalid_df



## 5. Define validation rules

# Status values are compared after trimming and converting to uppercase.
# The accepted values below follow the domain values configured for this
# Update them only when the business definition confirms a new value.

def normalized_status(column_name):
    return F.upper(F.trim(F.col(column_name)))


VALID_ORDER_STATUSES = [
    "DELIVERED",
    "SHIPPED",
    "PROCESSING",
    "CANCELLED"
]

VALID_PAYMENT_STATUSES = [
    "SUCCESS",
    "PENDING",
    "FAILED",
    "REFUNDED"
]

VALID_SHIPMENT_STATUSES = [
    "PENDING",
    "PROCESSING",
    "SHIPPED",
    "IN_TRANSIT",
    "OUT_FOR_DELIVERY",
    "DELIVERED",
    "CANCELLED",
    "RETURNED",
    "RETURN_TO_ORIGIN"
]

VALIDATION_RULES = {

    "customers": [
        (
            "MISSING_CUSTOMER_ID",
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == "")
        ),
        (
            "MISSING_FIRST_NAME",
            F.col("first_name").isNull()
            | (F.trim(F.col("first_name")) == "")
        ),
        (
            "MISSING_EMAIL",
            F.col("email").isNull()
            | (F.trim(F.col("email")) == "")
        ),
        (
            "INVALID_EMAIL",
            F.col("email").isNotNull()
            & (F.trim(F.col("email")) != "")
            & (
                ~F.trim(F.col("email")).rlike(
                    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
                )
            )
        ),
        (
            "MISSING_CREATED_DATE",
            F.col("created_at").isNull()
        )
    ],

    "products": [
        (
            "MISSING_PRODUCT_ID",
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == "")
        ),
        (
            "MISSING_PRODUCT_NAME",
            F.col("product_name").isNull()
            | (F.trim(F.col("product_name")) == "")
        ),
        (
            "INVALID_UNIT_PRICE",
            F.col("unit_price").isNull()
            | (F.col("unit_price") < 0)
        ),
        (
            "INVALID_UNIT_COST",
            F.col("unit_cost").isNull()
            | (F.col("unit_cost") < 0)
        ),
        (
            "MISSING_CREATED_DATE",
            F.col("created_at").isNull()
        )
    ],

    "orders": [
        (
            "MISSING_ORDER_ID",
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ),
        (
            "MISSING_CUSTOMER_ID",
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == "")
        ),
        (
            "MISSING_ORDER_TIMESTAMP",
            F.col("order_timestamp").isNull()
        ),
        (
            "INVALID_ORDER_STATUS",
            F.col("order_status").isNull()
            | (~normalized_status("order_status").isin(*VALID_ORDER_STATUSES))
        ),
        (
            "INVALID_ORDER_TOTAL",
            F.col("order_total").isNull()
            | (F.col("order_total") < 0)
        )
    ],

    "order_items": [
        (
            "MISSING_ORDER_ITEM_ID",
            F.col("order_item_id").isNull()
            | (F.trim(F.col("order_item_id")) == "")
        ),
        (
            "MISSING_ORDER_ID",
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ),
        (
            "MISSING_PRODUCT_ID",
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == "")
        ),
        (
            "INVALID_QUANTITY",
            F.col("quantity").isNull()
            | (F.col("quantity") <= 0)
        ),
        (
            "INVALID_UNIT_PRICE",
            F.col("unit_price").isNull()
            | (F.col("unit_price") < 0)
        ),
        (
            "INVALID_DISCOUNT",
            F.col("discount_rate").isNull()
            | (F.col("discount_rate") < 0)
            | (F.col("discount_rate") > 1)
        ),
        (
            "INVALID_LINE_TOTAL",
            F.col("line_total").isNull()
            | (F.col("line_total") < 0)
        )
    ],

    "payments": [
        (
            "MISSING_PAYMENT_ID",
            F.col("payment_id").isNull()
            | (F.trim(F.col("payment_id")) == "")
        ),
        (
            "MISSING_ORDER_ID",
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ),
        (
            "INVALID_PAYMENT_STATUS",
            F.col("payment_status").isNull()
            | (~normalized_status("payment_status").isin(*VALID_PAYMENT_STATUSES))
        ),
        (
            "INVALID_PAYMENT_AMOUNT",
            F.col("amount").isNull()
            | (F.col("amount") < 0)
        ),
        (
            "MISSING_PAYMENT_TIMESTAMP",
            F.col("payment_timestamp").isNull()
        )
    ],

    "shipments": [
        (
            "MISSING_SHIPMENT_ID",
            F.col("shipment_id").isNull()
            | (F.trim(F.col("shipment_id")) == "")
        ),
        (
            "MISSING_ORDER_ID",
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ),
        (
            "INVALID_SHIPMENT_STATUS",
            F.col("shipment_status").isNull()
            | (~normalized_status("shipment_status").isin(*VALID_SHIPMENT_STATUSES))
        ),
        (
            "DELIVERED_WITHOUT_DELIVERY_DATE",
            (normalized_status("shipment_status") == "DELIVERED")
            & F.col("delivered_date").isNull()
        ),
        (
            "DELIVERY_BEFORE_SHIPMENT",
            F.col("delivered_date").isNotNull()
            & F.col("shipped_date").isNotNull()
            & (F.col("delivered_date") < F.col("shipped_date"))
        )
    ],

    "inventory": [
        (
            "MISSING_INVENTORY_SNAPSHOT_ID",
            F.col("inventory_snapshot_id").isNull()
            | (F.trim(F.col("inventory_snapshot_id")) == "")
        ),
        (
            "MISSING_PRODUCT_ID",
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == "")
        ),
        (
            "MISSING_WAREHOUSE_ID",
            F.col("warehouse_id").isNull()
            | (F.trim(F.col("warehouse_id")) == "")
        ),
        (
            "INVALID_INVENTORY_QUANTITY",
            F.col("quantity_on_hand").isNull()
            | (F.col("quantity_on_hand") < 0)
        ),
        (
            "MISSING_SNAPSHOT_DATE",
            F.col("snapshot_date").isNull()
        )
    ]
}

print(f"Validation rules configured for {len(VALIDATION_RULES)} datasets.")


## 6. Read and validate all Silver datasets

valid_datasets = {}
invalid_datasets = {}
dataset_counts = {}

for dataset_name, config in DATASETS.items():

    source_table = f"{CATALOG}.{SILVER_SCHEMA}.{config['table']}"

    print("-" * 70)
    print(f"Processing: {source_table}")

    source_df = spark.table(source_table)

    valid_df, invalid_df = identify_invalid_records(
        source_df,
        dataset_name,
        VALIDATION_RULES[dataset_name]
    )

    valid_datasets[dataset_name] = valid_df
    invalid_datasets[dataset_name] = invalid_df

    total_count = source_df.count()
    valid_count = valid_df.count()
    invalid_count = invalid_df.count()

    dataset_counts[dataset_name] = {
        "total_records": total_count,
        "valid_records": valid_count,
        "invalid_records": invalid_count
    }

    if total_count != valid_count + invalid_count:
        raise Exception(
            f"Count reconciliation failed for {dataset_name}: "
            f"total={total_count}, valid={valid_count}, invalid={invalid_count}"
        )

    print(f"Total records  : {total_count}")
    print(f"Valid records  : {valid_count}")
    print(f"Invalid records: {invalid_count}")

print("-" * 70)
print("All Silver datasets validated and reconciled.")


## 7. Display invalid records by dataset

for dataset_name, invalid_df in invalid_datasets.items():

    print(f"Quarantined records: {dataset_name}")

    display(invalid_df)


## 8. Combine current invalid records

# The current quarantine table represents the latest validation snapshot,
# not an append-only event log.

invalid_dataframes = list(invalid_datasets.values())

if not invalid_dataframes:
    raise Exception("No datasets were configured for quarantine validation.")

combined_quarantine_df = invalid_dataframes[0]

for next_df in invalid_dataframes[1:]:
    combined_quarantine_df = combined_quarantine_df.unionByName(
        next_df,
        allowMissingColumns=True
    )

# Add a stable record key using the configured business key.

# If a business key is null, use a hash of the row payload as a fallback.

def add_quarantine_record_key(df, dataset_name, business_key):

    payload_columns = [
        c for c in df.columns
        if c not in {
            "_quarantine_record_key",
            "_quarantine_run_id",
            "_quarantine_run_timestamp",
            "_quarantined_at"
        }
    ]

    payload_json = F.to_json(
        F.struct(*[F.col(c) for c in payload_columns])
    )

    business_key_value = F.trim(F.col(business_key).cast("string"))

    fallback_key = F.sha2(
        F.concat_ws(
            "||",
            F.lit(dataset_name),
            F.coalesce(payload_json, F.lit(""))
        ),
        256
    )

    return df.withColumn(
        "_quarantine_record_key",
        F.when(
            F.col(business_key).isNotNull()
            & (business_key_value != ""),
            F.sha2(
                F.concat_ws(
                    "||",
                    F.lit(dataset_name),
                    business_key_value
                ),
                256
            )
        ).otherwise(fallback_key)
    )


keyed_current_dfs = []

for dataset_name, config in DATASETS.items():

    keyed_df = add_quarantine_record_key(
        invalid_datasets[dataset_name],
        dataset_name,
        config["business_key"]
    )

    keyed_current_dfs.append(keyed_df)

combined_quarantine_df = keyed_current_dfs[0]

for next_df in keyed_current_dfs[1:]:
    combined_quarantine_df = combined_quarantine_df.unionByName(
        next_df,
        allowMissingColumns=True
    )

current_invalid_count = combined_quarantine_df.count()

print(f"Current invalid records identified: {current_invalid_count}")


## 9. Refresh current quarantine snapshot

(
    combined_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Current quarantine snapshot saved: {QUARANTINE_TABLE}")


## 10. Maintain deduplicated quarantine history

# History is maintained as one latest record per dataset/business key.
# This is a unique-record history, not a record of every validation run.
# Existing history is retained and deduplicated; it is not dropped.

if spark.catalog.tableExists(QUARANTINE_HISTORY_TABLE):

    existing_history_df = spark.table(QUARANTINE_HISTORY_TABLE)

    # Add a key to legacy history rows if the key column did not exist.

    history_parts = []

    for dataset_name, config in DATASETS.items():

        if "_quarantine_dataset" not in existing_history_df.columns:
            raise Exception(
                "Existing quarantine history does not contain "
                "_quarantine_dataset. Inspect the table schema before continuing."
            )

        dataset_history = existing_history_df.filter(
            F.col("_quarantine_dataset") == dataset_name
        )

        if dataset_history.limit(1).count() > 0:

            dataset_history = add_quarantine_record_key(
                dataset_history,
                dataset_name,
                config["business_key"]
            )

            history_parts.append(dataset_history)

    if history_parts:

        existing_history_keyed = history_parts[0]

        for next_df in history_parts[1:]:
            existing_history_keyed = existing_history_keyed.unionByName(
                next_df,
                allowMissingColumns=True
            )

    else:
        existing_history_keyed = combined_quarantine_df.limit(0)

else:
    existing_history_keyed = combined_quarantine_df.limit(0)


# Current records take priority over old history when the same key exists.

existing_history_keyed = (
    existing_history_keyed
    .withColumn("_history_priority", F.lit(0))
)

current_history_keyed = (
    combined_quarantine_df
    .withColumn("_history_priority", F.lit(1))
)

history_candidates_df = existing_history_keyed.unionByName(
    current_history_keyed,
    allowMissingColumns=True
)

history_window = (
    Window
    .partitionBy(
        "_quarantine_dataset",
        "_quarantine_record_key"
    )
    .orderBy(
        F.col("_history_priority").desc(),
        F.col("_quarantine_run_timestamp").desc_nulls_last(),
        F.col("_quarantined_at").desc_nulls_last()
    )
)

deduplicated_history_df = (
    history_candidates_df
    .withColumn(
        "_history_row_number",
        F.row_number().over(history_window)
    )
    .filter(F.col("_history_row_number") == 1)
    .drop(
        "_history_row_number",
        "_history_priority"
    )
)

(
    deduplicated_history_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_HISTORY_TABLE)
)

print(f"Deduplicated quarantine history saved: {QUARANTINE_HISTORY_TABLE}")


## 11. Generate quarantine summary

# Each row is grouped by its complete combination of rule violations.
# For example, a record violating two rules appears in a combined reason group.

quarantine_summary_df = (
    combined_quarantine_df
    .groupBy(
        "_quarantine_dataset",
        "_quarantine_reason"
    )
    .agg(
        F.count("*").alias("quarantined_records")
    )
    .withColumn(
        "summary_generated_at",
        F.current_timestamp()
    )
    .withColumn(
        "quarantine_run_id",
        F.lit(RUN_ID)
    )
    .orderBy(
        "_quarantine_dataset",
        F.desc("quarantined_records")
    )
)

display(quarantine_summary_df)

(
    quarantine_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(VALIDATION_SUMMARY_TABLE)
)

print(f"Quarantine summary saved: {VALIDATION_SUMMARY_TABLE}")


## 12. Generate dataset-level statistics

statistics_rows = []

for dataset_name, counts in dataset_counts.items():

    total_count = counts["total_records"]
    valid_count = counts["valid_records"]
    invalid_count = counts["invalid_records"]

    statistics_rows.append((
        dataset_name,
        total_count,
        valid_count,
        invalid_count,
        round(valid_count / total_count * 100, 2) if total_count else 0.0,
        round(invalid_count / total_count * 100, 2) if total_count else 0.0,
        RUN_ID
    ))

statistics_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("total_records", LongType(), False),
    StructField("valid_records", LongType(), False),
    StructField("invalid_records", LongType(), False),
    StructField("valid_percentage", DoubleType(), False),
    StructField("invalid_percentage", DoubleType(), False),
    StructField("quarantine_run_id", StringType(), False)
])

statistics_df = spark.createDataFrame(
    statistics_rows,
    schema=statistics_schema
)

display(statistics_df)

(
    statistics_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DATASET_STATISTICS_TABLE)
)

print(f"Dataset statistics saved: {DATASET_STATISTICS_TABLE}")


## 13. Validate current quarantine and history uniqueness

current_quarantine_df = spark.table(QUARANTINE_TABLE)
history_df = spark.table(QUARANTINE_HISTORY_TABLE)

current_count = current_quarantine_df.count()
history_count = history_df.count()

duplicate_history_keys = (
    history_df
    .groupBy(
        "_quarantine_dataset",
        "_quarantine_record_key"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_current_keys = (
    current_quarantine_df
    .groupBy(
        "_quarantine_dataset",
        "_quarantine_record_key"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Current quarantine rows       : {current_count}")
print(f"Historical unique records     : {history_count}")
print(f"Duplicate current keys        : {duplicate_current_keys}")
print(f"Duplicate historical keys     : {duplicate_history_keys}")

if duplicate_history_keys > 0:
    raise Exception(
        f"History uniqueness validation failed: {duplicate_history_keys} duplicate keys."
    )

if duplicate_current_keys > 0:
    print(
        "WARNING: Current snapshot contains repeated business keys. "
        "Inspect the source data and business-key uniqueness."
    )


## 14. Final reconciliation

total_source_records = sum(
    counts["total_records"]
    for counts in dataset_counts.values()
)

total_valid_records = sum(
    counts["valid_records"]
    for counts in dataset_counts.values()
)

total_invalid_records = sum(
    counts["invalid_records"]
    for counts in dataset_counts.values()
)

if total_source_records != total_valid_records + total_invalid_records:
    raise Exception("Final source/valid/invalid reconciliation failed.")

if current_count != total_invalid_records:
    raise Exception(
        "Current quarantine count does not match the invalid-record total: "
        f"current={current_count}, expected={total_invalid_records}"
    )

print("=" * 70)
print("QUARANTINE MANAGEMENT COMPLETED")
print("=" * 70)
print(f"Run ID                         : {RUN_ID}")
print(f"Datasets processed             : {len(DATASETS)}")
print(f"Total Silver source records    : {total_source_records}")
print(f"Valid records                  : {total_valid_records}")
print(f"Invalid records                : {total_invalid_records}")
print(f"Current quarantine records     : {current_count}")
print(f"Historical unique records      : {history_count}")
print(f"Duplicate current keys         : {duplicate_current_keys}")
print(f"Duplicate history keys         : {duplicate_history_keys}")
print(f"Current quarantine table       : {QUARANTINE_TABLE}")
print(f"History table                  : {QUARANTINE_HISTORY_TABLE}")
print(f"Quarantine summary table       : {VALIDATION_SUMMARY_TABLE}")
print(f"Dataset statistics table       : {DATASET_STATISTICS_TABLE}")
print("=" * 70)

display(
    current_quarantine_df
    .orderBy("_quarantine_dataset", "_quarantine_reason")
)


## 15. Final dataset statistics

display(
    spark.table(DATASET_STATISTICS_TABLE)
    .orderBy("dataset_name")
)


## 16. Final quarantine reason summary

display(
    spark.table(VALIDATION_SUMMARY_TABLE)
    .orderBy(
        "_quarantine_dataset",
        F.desc("quarantined_records")
    )
)



## Quarantine Management Tables:


| Table | Purpose |
|---|---|
| `quarantine_records` | Stores invalid records identified during the latest execution. |
| `quarantine_history` | Maintains invalid records from previous and current executions for historical tracking. |
| `quarantine_summary` | Stores failure counts grouped by dataset and failure reason. |


### Storage Location:


All three tables are stored in the Silver schema:

`workspace.ecommerce_silver`